# LendingClub Loan Default Prediction

Predicting whether a loan applicant will default (`Charged Off`) or repay (`Fully Paid`), using only information available **at application time**.

This is a modernized rewrite of the classic Kaggle notebook [*Lending Club Loan Defaulters Prediction*](https://www.kaggle.com/code/faressayah/lending-club-loan-defaulters-prediction).

**What changed, and why:**

| Area | Original | Here |
|---|---|---|
| Structure | All inline | Reusable `src/` package; this notebook is the narrative |
| Zip codes | One-hot encoded raw zip (hundreds of sparse columns) | 2-letter `state` |
| Validation | Test set used as Keras validation set | Genuine train/val/test; test touched once |
| Models | One Keras MLP | XGBoost + LightGBM vs. a modernized MLP |
| Metrics | Accuracy/F1 @ 0.5 | ROC-AUC, PR-AUC, KS, Brier + cost-based threshold |
| Features | Raw columns | + loan-to-income, installment burden, utilization ratios |
| Explainability | None | SHAP + gain-based importance |
| Artifacts | None saved | Preprocessor, models, metadata persisted |

> **Note:** every heavy step here is also available as a script — `python -m src.train` runs this whole pipeline and writes `reports/REPORT.md` plus all figures. Use this notebook to understand the reasoning; use the script to reproduce results.

## 1. Business framing

Two errors, with very different costs:

- **Approving a borrower who defaults** → lose principal
- **Rejecting a borrower who would have repaid** → lose the interest margin

These are not symmetric, which is why a 0.5 probability cut-off is the wrong default. We'll choose the threshold explicitly from a cost assumption later.

## 2. Setup

In [ ]:
import sys
sys.path.insert(0, "..")     # so `import src` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)
%matplotlib inline

from src import config, data, features, preprocessing, models, evaluate, tuning

### Loading the data (offline)

Download `lending_club_loan_two.csv` from the Kaggle notebook linked above and place it in `data/raw/`. Optionally also grab `lending_club_info.csv` for column descriptions.

Verify from the terminal anytime with `python -m src.data`.

In [ ]:
df = data.load_raw_data()
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df["loan_status"].value_counts(normalize=True)

About 80/20. That imbalance is the single most important fact about this dataset — it's why accuracy is a misleading metric here (a model that approves everyone scores ~80%).

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["loan_status"].value_counts().plot(kind="bar", ax=axes[0], color=["#3b7dd8", "#d84f3b"])
axes[0].set_title("Target distribution")
axes[0].tick_params(axis="x", rotation=0)

default_by_grade = (df.assign(is_default=(df.loan_status == "Charged Off"))
                      .groupby("grade")["is_default"].mean())
default_by_grade.plot(kind="bar", ax=axes[1], color="#d84f3b")
axes[1].set(title="Default rate by LendingClub grade", ylabel="default rate")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

Default rate climbs steadily A → G. LendingClub's own grading already carries real signal, which sets the bar our model has to beat.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), cmap="viridis", center=0)
plt.title("Correlation matrix (numeric features)")
plt.show()

`loan_amnt` and `installment` are nearly perfectly correlated — `installment` is a deterministic function of amount, rate and term. Tree models tolerate this fine, so we keep both, but it's worth knowing if you ever swap in a linear model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["int_rate", "dti", "annual_inc"]):
    for status, color in [("Fully Paid", "#3b7dd8"), ("Charged Off", "#d84f3b")]:
        subset = df.loc[df.loan_status == status, col]
        if col == "annual_inc":
            subset = subset[subset < subset.quantile(0.99)]   # trim extreme tail
        ax.hist(subset, bins=40, alpha=0.55, label=status, color=color, density=True)
    ax.set_xlabel(col); ax.legend()
plt.suptitle("Feature distributions by outcome")
plt.tight_layout(); plt.show()

Higher interest rates and higher debt-to-income skew toward charge-off. Income is heavily right-skewed — a good argument for the log transform we add during feature engineering.

In [ ]:
# Missing values worth knowing about before cleaning
missing = df.isna().sum()
missing = (missing[missing > 0]
           .to_frame("n_missing")
           .assign(pct=lambda d: d.n_missing / len(df) * 100)
           .sort_values("pct", ascending=False))
missing

## 4. Cleaning & feature engineering

All logic lives in `src/features.py`. Summary of decisions:

| Step | Why |
|---|---|
| Drop `issue_d` | **Leakage** — not known at application time |
| Drop `emp_title`, `emp_length`, `title`, `grade` | Too high-cardinality / near-zero signal / redundant with `sub_grade` and `purpose` |
| `address` → `state` | The original one-hot encoded raw zip codes into hundreds of sparse columns; state is compact and generalizes |
| `mort_acc` imputation | Filled from the `total_acc`-grouped mean, **fit on train only** |
| `pub_rec`, `pub_rec_bankruptcies` | Binarized (0 vs 1+) — the raw counts are long-tailed and sparse |
| `home_ownership` | `ANY`/`NONE` merged into `OTHER` |
| `term`, `earliest_cr_line` | Parsed to numeric (months, year) |
| **New ratio features** | `loan_to_income`, `annual_installment_to_income`, `revol_bal_to_income`, `open_to_total_acc_ratio`, `log_annual_inc` — trees can't easily synthesize ratios from raw columns, and "obligation relative to capacity" is the core underwriting question |

In [ ]:
X, y = features.clean_and_engineer(df)
numeric_cols, categorical_cols = features.get_feature_lists(X)

print(f"Rows: {len(X):,}   Default rate: {y.mean():.2%}")
print(f"Features: {X.shape[1]} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
X.head()

## 5. Train / validation / test split

The original notebook passed the test set as Keras' `validation_data`, so early stopping and model selection effectively saw the test data. Here the split is three-way: we tune and pick thresholds on **validation**, and touch **test** exactly once at the end.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=config.TEST_SIZE + config.VAL_SIZE,
    random_state=config.RANDOM_STATE, stratify=y)

val_frac = config.VAL_SIZE / (config.TEST_SIZE + config.VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1 - val_frac,
    random_state=config.RANDOM_STATE, stratify=y_temp)

print(f"train={len(X_train):,}   val={len(X_val):,}   test={len(X_test):,}")
print(f"default rate -> train {y_train.mean():.2%} | val {y_val.mean():.2%} | test {y_test.mean():.2%}")

In [ ]:
preprocessor = preprocessing.build_preprocessor(numeric_cols, categorical_cols)

X_train_t = preprocessor.fit_transform(X_train)   # fit on TRAIN only
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)
feature_names = preprocessing.get_output_feature_names(preprocessor)

print(f"Encoded features: {len(feature_names)}")

## 6. Models

Class imbalance is handled with `scale_pos_weight` (trees) and `class_weight` (MLP), rather than SMOTE or undersampling — no synthetic rows, no discarded data.

In [ ]:
scale_pos_weight = models.compute_scale_pos_weight(y_train)
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.2f}")

val_proba, test_proba, fitted = {}, {}, {}

### 6.1 Optional: hyperparameter tuning

Set `RUN_TUNING = True` to search with Optuna (falls back to `RandomizedSearchCV` if Optuna isn't installed). It optimizes **PR-AUC**, searches on a stratified subsample for speed, and refits the winner on the full training set.

This takes several minutes — leave it off for a first pass.

In [ ]:
RUN_TUNING = False      # flip to True to search
best_params = {"xgboost": {}, "lightgbm": {}}

if RUN_TUNING:
    for name in ("xgboost", "lightgbm"):
        best_params[name] = tuning.tune_model(name, X_train_t, y_train,
                                              scale_pos_weight, n_trials=30)
        print(f"{name}: {best_params[name]}")

### 6.2 XGBoost

In [ ]:
xgb_model = models.build_xgboost(scale_pos_weight, best_params["xgboost"])
xgb_model.fit(X_train_t, y_train, eval_set=[(X_val_t, y_val)], verbose=False)

fitted["XGBoost"] = xgb_model
val_proba["XGBoost"]  = xgb_model.predict_proba(X_val_t)[:, 1]
test_proba["XGBoost"] = xgb_model.predict_proba(X_test_t)[:, 1]
print("done")

### 6.3 LightGBM

In [ ]:
lgb_model = models.build_lightgbm(scale_pos_weight, best_params["lightgbm"])
models.fit_lightgbm(lgb_model, X_train_t, y_train, X_val_t, y_val)

fitted["LightGBM"] = lgb_model
val_proba["LightGBM"]  = lgb_model.predict_proba(X_val_t)[:, 1]
test_proba["LightGBM"] = lgb_model.predict_proba(X_test_t)[:, 1]
print("done")

### 6.4 Keras MLP (modernized baseline)

Requires `pip install tensorflow`. Skips cleanly if it isn't installed — the boosted trees are the centerpiece.

In [ ]:
mlp_history = None
try:
    mlp = models.build_keras_mlp(X_train_t.shape[1])
    mlp_history = mlp.fit(
        X_train_t, y_train, validation_data=(X_val_t, y_val),
        epochs=config.MLP_EPOCHS, batch_size=config.MLP_BATCH_SIZE,
        class_weight={0: 1.0, 1: scale_pos_weight},
        callbacks=models.get_keras_callbacks(), verbose=0)

    fitted["Keras MLP"] = mlp
    val_proba["Keras MLP"]  = mlp.predict(X_val_t,  verbose=0).ravel()
    test_proba["Keras MLP"] = mlp.predict(X_test_t, verbose=0).ravel()
    print(f"Trained {len(mlp_history.history['loss'])} epochs (early stopping).")
except ImportError:
    print("TensorFlow not installed - skipping the MLP baseline.")

In [ ]:
if mlp_history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(mlp_history.history["loss"], label="train")
    axes[0].plot(mlp_history.history["val_loss"], label="val")
    axes[0].set(xlabel="Epoch", ylabel="Binary cross-entropy", title="Loss")
    axes[0].legend()
    if "pr_auc" in mlp_history.history:
        axes[1].plot(mlp_history.history["pr_auc"], label="train")
        axes[1].plot(mlp_history.history["val_pr_auc"], label="val")
        axes[1].set(xlabel="Epoch", ylabel="PR-AUC", title="PR-AUC")
        axes[1].legend()
    plt.tight_layout(); plt.show()

## 7. Choosing an operating threshold

This is the step the original notebook skipped entirely.

A lender doesn't care about the 0.5 cut-off; it cares about expected loss. We assume a false negative (approving a defaulter) costs several times more than a false positive (rejecting a good borrower), then sweep thresholds to minimize expected cost — **on the validation set**, so the test set stays clean.

One important guard: pure cost minimization under an expensive FN will happily reject nearly everyone (rejecting all applicants has zero default losses, and also zero business). So we constrain the search to thresholds that still approve at least `MIN_APPROVAL_RATE` of applicants.

In [ ]:
print(f"Cost assumptions: FN = {config.COST_FALSE_NEGATIVE}x, FP = {config.COST_FALSE_POSITIVE}x")
print(f"Constraint: approve at least {config.MIN_APPROVAL_RATE:.0%} of applicants\n")

thresholds = {}
for name in fitted:
    thr, cost = evaluate.find_optimal_threshold(y_val, val_proba[name])
    thresholds[name] = thr
    print(f"{name:<12} threshold={thr:.3f}   val cost/applicant={cost:.4f}")

In [ ]:
# What the threshold trade-off actually looks like
first_model = list(fitted)[0]
evaluate.plot_threshold_sweep(y_val, val_proba[first_model],
                              chosen_threshold=thresholds[first_model])
plt.show()

## 8. Evaluation on the held-out test set

Primary metrics are threshold-free:

- **ROC-AUC** — overall ranking quality
- **PR-AUC** — more informative than ROC-AUC under imbalance
- **KS** — the standard credit-scoring separation metric (>0.3 is generally considered usable)
- **Brier** — probability calibration quality (lower is better)

In [ ]:
rows = []
for name in fitted:
    rows.append(evaluate.score_summary(y_test, test_proba[name],
                                       threshold=thresholds[name], label=name))

comparison = pd.DataFrame(rows)
comparison[["model", "roc_auc", "pr_auc", "ks_stat", "brier",
            "threshold", "precision", "recall", "f1", "approval_rate"]]

In [ ]:
best_name = comparison.loc[comparison["roc_auc"].idxmax(), "model"]
print(f"Best model by ROC-AUC: {best_name}")

plot_set = {name: (y_test, test_proba[name]) for name in fitted}
evaluate.plot_roc_pr_curves(plot_set); plt.show()

In [ ]:
evaluate.plot_ks_curve(y_test, test_proba[best_name], label=best_name); plt.show()

### Compare against the naive 0.5 threshold

This is the concrete payoff of doing threshold selection at all.

In [ ]:
naive = pd.DataFrame([
    evaluate.score_summary(y_test, test_proba[best_name], threshold=0.5,
                           label=f"{best_name} @ 0.5", verbose=False),
    evaluate.score_summary(y_test, test_proba[best_name], threshold=thresholds[best_name],
                           label=f"{best_name} @ cost-optimal", verbose=False),
])
naive[["model", "threshold", "precision", "recall", "f1",
       "expected_cost", "approval_rate"]]

## 9. Calibration

`scale_pos_weight` deliberately distorts predicted probabilities to fight imbalance. That's fine for *ranking* applicants, but wrong if you want to read the output as a real probability of default (e.g. to compute expected loss = PD × exposure).

Isotonic regression fitted on the validation set fixes the scale without retraining.

In [ ]:
calibrated = None
if best_name in ("XGBoost", "LightGBM"):
    calibrated = models.calibrate(fitted[best_name], X_val_t, y_val, method="isotonic")
    cal_proba = calibrated.predict_proba(X_test_t)[:, 1]

    evaluate.plot_calibration({
        f"{best_name} (raw)": (y_test, test_proba[best_name]),
        f"{best_name} (calibrated)": (y_test, cal_proba),
    })
    plt.show()

    _ = evaluate.score_summary(y_test, cal_proba, threshold=thresholds[best_name],
                               label=f"{best_name} (calibrated)")

## 10. Risk deciles

How a credit team actually sanity-checks a scorecard: sort applicants by predicted risk, bucket into ten groups, and check the observed default rate climbs monotonically.

In [ ]:
deciles = evaluate.decile_table(y_test, test_proba[best_name])
display(deciles)

evaluate.plot_decile_chart(deciles); plt.show()

## 11. Explainability

Two complementary views: native gain-based importance (fast, global) and SHAP (directional — shows *which way* each feature pushes risk).

In [ ]:
if hasattr(fitted[best_name], "feature_importances_"):
    evaluate.plot_feature_importance(fitted[best_name], feature_names, top_n=20)
    plt.show()

In [ ]:
tree_model = fitted[best_name] if best_name in ("XGBoost", "LightGBM") else fitted["XGBoost"]
n = min(config.SHAP_SAMPLE_SIZE, len(X_test_t))
evaluate.shap_summary(tree_model, X_test_t[:n], feature_names)
plt.show()

Read this as: each dot is one applicant, position on the x-axis is how much that feature pushed their predicted risk up (right) or down (left), and colour is the feature's value. A red cluster on the right means "high values of this feature increase predicted default risk".

## 12. Save artifacts

In [ ]:
import joblib, json

joblib.dump(preprocessor, config.MODELS_DIR / "preprocessor.joblib")
joblib.dump(fitted[best_name] if calibrated is None else calibrated,
            config.MODELS_DIR / "best_model.joblib")

metadata = {
    "best_model": best_name,
    "calibrated": calibrated is not None,
    "threshold": float(thresholds[best_name]),
    "n_features_encoded": len(feature_names),
    "default_rate": float(y.mean()),
}
(config.MODELS_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"Saved to {config.MODELS_DIR}")

## 13. Conclusions

- **Grade/sub-grade and interest rate dominate**, which is reassuring — LendingClub's own pricing already encodes most of the risk signal, and the model recovers that rather than contradicting it.
- **The engineered ratio features earn their place** — loan-to-income and installment burden typically rank high in SHAP, above many raw columns.
- **Gradient-boosted trees match or beat the neural net** while training in a fraction of the time and needing far less tuning. On tabular data this is the usual outcome, and here it's demonstrated rather than assumed.
- **Threshold choice matters as much as model choice.** Moving from 0.5 to a cost-optimal cut-off changes precision/recall materially without changing the model at all.
- **Calibration is a separate concern from discrimination.** ROC-AUC barely moves after isotonic calibration, but the Brier score improves — the ranking was already good, the probability *scale* was off.

### Honest limitations

- This dataset contains only **funded** loans. Applicants LendingClub rejected outright are absent, so the model learns "who defaults among those already approved", not "who defaults among all applicants". That's survivorship bias, and it limits real-world deployment.
- Loan outcomes are correlated with macroeconomic conditions over time. A random split (used here, following the original) leaks some temporal information; a time-based split would give a more honest estimate of forward performance.
- `state` is used as a feature. In a real lending context, geography can proxy for protected characteristics and would need fair-lending review.

### Next steps

- Re-run with a **time-based split** on `issue_d` to measure temporal generalization
- Hyperparameter search (`RUN_TUNING = True` above, or `python -m src.train --tune`)
- Serve the model: `uvicorn src.serve:app --reload` → http://127.0.0.1:8000/docs
